## Notebook 10 — Media Behavior Baseline and Cluster Deviations

This notebook analyzes media behavior patterns across the synthetic U.S. population generated in the previous stages of the Market Kinetics pipeline.

The objective is to identify **media behaviors that characterize each structural cluster** by comparing cluster-level averages against the **national media baseline**.

The analysis proceeds in three steps:

1. Load the individual-level population dataset enriched with projected media behaviors.
2. Compute the **national media baseline** using ACS population weights.
3. Calculate **cluster-level deviations from the national baseline** in order to identify media behaviors that meaningfully distinguish each cluster.

These deviations will later be integrated with the psychological cluster signatures derived in Notebook 08 to construct final audience archetypes.

In [1]:
#import
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# Project root (assumes notebook is inside /notebooks)
PROJECT_ROOT   = Path().resolve().parent
DATA_MODELS    = PROJECT_ROOT / "data" / "societal_models"
DATA_PROCESSED = PROJECT_ROOT / "data" / "societal_processed"

In [3]:
df = pd.read_parquet(DATA_MODELS / "09_mk_structural_media_population.parquet")

In [4]:
media_cols = [
    "internet_access",
    "mobile_internet_access",
    "internet_frequency",
    "radio_listener",
    "youtube",
    "facebook",
    "instagram",
    "tiktok",
    "whatsapp",
    "reddit",
    "snapchat",
    "x_twitter",
    "threads",
    "bluesky",
    "truthsocial"
]

In [5]:
# computing national media baseline
national_media_baseline = (
    df[media_cols]
    .multiply(df["pwgtp"], axis=0)
    .sum()
    / df["pwgtp"].sum()
)

national_media_baseline

internet_access           0.946541
mobile_internet_access    0.939734
internet_frequency        1.772246
radio_listener            0.681662
youtube                   0.851719
facebook                  0.729095
instagram                 0.505528
tiktok                    0.401325
whatsapp                  0.331104
reddit                    0.267975
snapchat                  0.291080
x_twitter                 0.206566
threads                   0.091674
bluesky                   0.039250
truthsocial               0.034554
dtype: float64

In [6]:
# reshaping for visualization
baseline_df = (
    national_media_baseline
    .reset_index()
)

baseline_df.columns = ["media_trait", "national_baseline"]

baseline_df

,media_trait,national_baseline
0,internet_access,0.946541
1,mobile_internet_access,0.939734
2,internet_frequency,1.772246
3,radio_listener,0.681662
4,youtube,0.851719
5,facebook,0.729095
6,instagram,0.505528
7,tiktok,0.401325
8,whatsapp,0.331104
9,reddit,0.267975


In [7]:
baseline_path = DATA_PROCESSED / "10_mk_media_national_baseline.csv"

baseline_df.to_csv(baseline_path, index=False)

print("Saved:", baseline_path)

Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/10_mk_media_national_baseline.csv


In [8]:
# computing cluster media means weighted by population
cluster_media_means = (
    df
    .groupby("cluster")
    .apply(
        lambda g: (
            g[media_cols]
            .multiply(g["pwgtp"], axis=0)
            .sum()
            / g["pwgtp"].sum()
        )
    )
)

cluster_media_means

,internet_access,mobile_internet_access,internet_frequency,radio_listener,youtube,facebook,instagram,tiktok,whatsapp,reddit,snapchat,x_twitter,threads,bluesky,truthsocial
cluster,,,,,,,,,,,,,,,
0,0.970491,0.973391,1.617638,0.680902,0.903714,0.775802,0.586039,0.465436,0.401177,0.294924,0.335676,0.220894,0.106522,0.041183,0.030567
1,0.920193,0.907440,1.928869,0.698808,0.786632,0.707023,0.432250,0.337115,0.288757,0.228112,0.231955,0.185754,0.081926,0.035000,0.033168
2,0.950075,0.957057,1.722057,0.638858,0.883830,0.733446,0.568766,0.490706,0.415626,0.258154,0.338817,0.217875,0.104970,0.035043,0.033654
3,0.891648,0.857846,2.187705,0.734055,0.713229,0.605674,0.257814,0.200646,0.223496,0.146185,0.113486,0.140610,0.045391,0.026637,0.035797
4,0.945006,0.941502,1.739872,0.652595,0.874441,0.712306,0.528699,0.426348,0.330683,0.291925,0.321709,0.224159,0.084647,0.040415,0.036431
5,0.958212,0.962636,1.593589,0.617461,0.908475,0.766799,0.626650,0.533256,0.383599,0.326890,0.417186,0.243805,0.126974,0.042160,0.024738
6,0.974596,0.966821,1.681374,0.716830,0.880892,0.770074,0.529869,0.385059,0.311357,0.296582,0.284588,0.209331,0.092042,0.047425,0.044679


In [9]:
# compute deviations from national baseline
cluster_media_deviation = cluster_media_means - national_media_baseline

In [10]:
cluster_media_dev_long = (
    cluster_media_deviation
    .reset_index()
    .melt(
        id_vars="cluster",
        var_name="trait",
        value_name="deviation"
    )
)

cluster_media_dev_long.head()

,cluster,trait,deviation
0,0,internet_access,0.023950
1,1,internet_access,-0.026348
2,2,internet_access,0.003534
3,3,internet_access,-0.054892
4,4,internet_access,-0.001535


In [11]:
cluster_media_dev_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   cluster    105 non-null    uint16 
 1   trait      105 non-null    str    
 2   deviation  105 non-null    float64
dtypes: float64(1), str(1), uint16(1)
memory usage: 3.0 KB


In [12]:
cluster_media_dev_long["deviation"].describe()

count    105.000000
mean      -0.000183
std        0.077861
min       -0.247715
25%       -0.026348
50%        0.002765
75%        0.030629
max        0.415459
Name: deviation, dtype: float64

In [13]:
# based on the distribution of deviations, we can set a threshold for "significant" deviation (3%)
threshold = 0.03

cluster_media_signature = (
    cluster_media_dev_long
    .loc[cluster_media_dev_long["deviation"].abs() >= threshold]
    .copy()
)

In [14]:
# sorting
cluster_media_signature["abs_dev"] = cluster_media_signature["deviation"].abs()

cluster_media_signature.sort_values(
    ["cluster","abs_dev"],
    ascending=[True,False],
    inplace=True
)
cluster_media_signature

,cluster,trait,deviation,abs_dev
14,0,internet_frequency,-0.154608,0.154608
42,0,instagram,0.080511,0.080511
56,0,whatsapp,0.070073,0.070073
49,0,tiktok,0.064111,0.064111
28,0,youtube,0.051995,0.051995
35,0,facebook,0.046707,0.046707
70,0,snapchat,0.044596,0.044596
7,0,mobile_internet_access,0.033657,0.033657
15,1,internet_frequency,0.156622,0.156622
43,1,instagram,-0.073279,0.073279


### Media Behavior Deviations by Cluster

In this step, media behavior patterns were analyzed to identify platform usage characteristics that distinguish each structural cluster from the national population baseline.

First, a **national media baseline** was computed using the PWGTP population weights from the ACS-enriched synthetic population dataset. This baseline represents the estimated share of the U.S. population using each media platform or exhibiting each media behavior.

Second, **cluster-level weighted means** were calculated for all media variables. Deviations were then computed as:

cluster_mean − national_baseline

This produced a measure of **behavioral over- or under-indexing** for each cluster relative to the national population.

To isolate meaningful signals and remove noise, a **±0.03 deviation threshold (3 percentage points)** was applied, consistent with the filtering logic used previously for psychological traits. Only media behaviors exceeding this threshold were retained as **cluster-defining media signals**.

### Outcome

The resulting table captures the **media behavioral signature of each cluster**, highlighting platforms and behaviors where clusters significantly diverge from the national baseline. These signals reveal distinct patterns of digital engagement across clusters, such as stronger usage of specific social media platforms or lower overall internet activity.

These media signatures will be combined with the previously derived **psychological cluster signatures** to construct the final **audience archetypes** in the next stage of the analysis.

In [15]:
# for better readability, we can combine the trait name with the media usage prefix
cluster_media_signature["trait"] = "media_usage: " + cluster_media_signature["trait"]

In [16]:
# adding direction
cluster_media_signature["direction"] = np.where(
    cluster_media_signature["deviation"] > 0,
    "above baseline",
    "below baseline"
)

In [17]:
# align with psych traits
cluster_media_signature = cluster_media_signature[
    ["cluster","trait","deviation","direction"]
]

In [18]:
cluster_media_signature.head()

,cluster,trait,deviation,direction
14,0,media_usage: internet_frequency,-0.154608,below baseline
42,0,media_usage: instagram,0.080511,above baseline
56,0,media_usage: whatsapp,0.070073,above baseline
49,0,media_usage: tiktok,0.064111,above baseline
28,0,media_usage: youtube,0.051995,above baseline


In [19]:
psych_path = DATA_MODELS / "08_mk_cluster_psychological_signatures.parquet"

cluster_psych_signature = pd.read_parquet(psych_path)[
    ["cluster", "trait", "deviation", "direction"]
].copy()

print(cluster_psych_signature.shape)
cluster_psych_signature.head()

(30, 4)


,cluster,trait,deviation,direction
0,0,media_engagement: low,0.0322,above baseline
1,1,media_engagement: low,-0.0433,below baseline
2,1,party_alignment: democrat,0.0347,above baseline
3,1,media_engagement: high,0.0336,above baseline
4,1,religiosity: none,-0.0306,below baseline


In [20]:
cluster_psych_signature = cluster_psych_signature.copy()
cluster_media_signature = cluster_media_signature.copy()

cluster_psych_signature["source"] = "psych"
cluster_media_signature["source"] = "media"

In [21]:
# merging media and psych signatures
cluster_full_signature = pd.concat(
    [cluster_psych_signature, cluster_media_signature],
    ignore_index=True
)

In [22]:
# sort by signal strength
cluster_full_signature["abs_dev"] = cluster_full_signature["deviation"].abs()

cluster_full_signature.sort_values(
    ["cluster", "abs_dev"],
    ascending=[True, False],
    inplace=True
)

In [23]:
cluster_full_signature.head(20)

,cluster,trait,deviation,direction,source,abs_dev
30,0,media_usage: internet_frequency,-0.154608,below baseline,media,0.154608
31,0,media_usage: instagram,0.080511,above baseline,media,0.080511
32,0,media_usage: whatsapp,0.070073,above baseline,media,0.070073
33,0,media_usage: tiktok,0.064111,above baseline,media,0.064111
34,0,media_usage: youtube,0.051995,above baseline,media,0.051995
35,0,media_usage: facebook,0.046707,above baseline,media,0.046707
36,0,media_usage: snapchat,0.044596,above baseline,media,0.044596
37,0,media_usage: mobile_internet_access,0.033657,above baseline,media,0.033657
0,0,media_engagement: low,0.032200,above baseline,psych,0.032200
38,1,media_usage: internet_frequency,0.156622,above baseline,media,0.156622


In [24]:
# ANALYSIS

In [25]:
# selecting top traits per cluster and source
top_psych = (
    cluster_full_signature
    .query("source == 'psych'")
    .groupby("cluster")
    .head(3)
)

top_media = (
    cluster_full_signature
    .query("source == 'media'")
    .groupby("cluster")
    .head(3)
)

In [26]:
top_signals = pd.concat(
    [top_psych, top_media],
    ignore_index=True
)

In [27]:
top_signals.head(20)

,cluster,trait,deviation,direction,source,abs_dev
0,0,media_engagement: low,0.032200,above baseline,psych,0.032200
1,1,media_engagement: low,-0.043300,below baseline,psych,0.043300
2,1,party_alignment: democrat,0.034700,above baseline,psych,0.034700
3,1,media_engagement: high,0.033600,above baseline,psych,0.033600
4,2,media_engagement: low,0.077100,above baseline,psych,0.077100
5,2,media_engagement: high,-0.068600,below baseline,psych,0.068600
6,2,party_alignment: republican,-0.049600,below baseline,psych,0.049600
7,3,media_engagement: low,-0.138000,below baseline,psych,0.138000
8,3,media_engagement: high,0.119800,above baseline,psych,0.119800
9,3,ideology: conservative,0.080500,above baseline,psych,0.080500


In [28]:
# sort for readability
top_signals = top_signals.sort_values(
    ["cluster", "abs_dev"],
    ascending=[True, False]
)

In [29]:
# clean summary table
cluster_summary = (
    top_signals
    .groupby("cluster")["trait"]
    .apply(list)
    .reset_index(name="key_traits")
)

cluster_summary

,cluster,key_traits
0,0,"[media_usage: internet_frequency, media_usage:..."
1,1,"[media_usage: internet_frequency, media_usage:..."
2,2,"[media_usage: tiktok, media_usage: whatsapp, m..."
3,3,"[media_usage: internet_frequency, media_usage:..."
4,4,"[media_engagement: high, media_engagement: low..."
5,5,"[media_usage: internet_frequency, media_usage:..."
6,6,"[media_usage: internet_frequency, media_usage:..."


### Final Segmentation Table — Full Pipeline Output

This table combines structural population shares, psychological
signatures, and media signatures into a single cluster profile
for each of the 7 adult archetypes. This is the final output
of the societal baseline pipeline and the input to the MK Intel
TA analysis layer.

In [30]:
cluster_pop_path = DATA_MODELS / "06b_mk_structural_population_clustered_v2.parquet"

df_struct = pd.read_parquet(cluster_pop_path)

print("Shape:", df_struct.shape)
df_struct.head()

Shape: (778466, 15)


,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,mar_tier,tenure,household_size,vehicle_count,household_type,serialno,sporder,pwgtp,cluster
0,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,No_Rent,2,0,housing_unit,2023HU1043211,2,58,5
1,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,No_Rent,4,2,housing_unit,2019HU1076190,2,46,0
2,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,Never_Married,Group_Quarters,1,0,group_quarters,2019GQ0046130,1,12,1
3,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Owner,5,2,housing_unit,2019HU0403832,1,76,0
4,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,No_Rent,4,1,housing_unit,2019HU0277198,1,64,5


In [31]:
# computing weighted cluster population
cluster_population = (
    df_struct
    .groupby("cluster")["pwgtp"]
    .sum()
    .reset_index(name="weighted_population")
)

cluster_population

,cluster,weighted_population
0,0,5327381
1,1,4997323
2,2,1652705
3,3,3872944
4,4,4659958
5,5,4460229
6,6,5914104


In [32]:
# compute population share
cluster_population["population_share"] = (
    cluster_population["weighted_population"]
    / cluster_population["weighted_population"].sum()
)

In [33]:
# keeping final fields
cluster_population = cluster_population[
    ["cluster", "population_share"]
]

cluster_population

,cluster,population_share
0,0,0.172493
1,1,0.161806
2,2,0.053512
3,3,0.125400
4,4,0.150883
5,5,0.144416
6,6,0.191490


In [34]:
# check
cluster_population["population_share"].sum()

np.float64(1.0000000000000002)

In [35]:
# create psych table
psych_traits = (
    top_signals
    .query("source == 'psych'")
    .assign(trait_with_dir=lambda df: df.apply(
        lambda r: f"{r['trait']} ({r['direction']})", axis=1
    ))
    .groupby("cluster")["trait_with_dir"]
    .apply(list)
    .reset_index(name="psych_signals")
)

In [36]:
# create media table
# include direction in trait label for consistency
media_traits = (
    top_signals
    .query("source == 'media'")
    .assign(trait_with_dir=lambda df: df.apply(
        lambda r: f"{r['trait']} ({r['direction']})", axis=1
    ))
    .groupby("cluster")["trait_with_dir"]
    .apply(list)
    .reset_index(name="media_signals")
)

media_traits

,cluster,media_signals
0,0,[media_usage: internet_frequency (below baseli...
1,1,[media_usage: internet_frequency (above baseli...
2,2,"[media_usage: tiktok (above baseline), media_u..."
3,3,[media_usage: internet_frequency (above baseli...
4,4,[media_usage: internet_frequency (below baseli...
5,5,[media_usage: internet_frequency (below baseli...
6,6,[media_usage: internet_frequency (below baseli...


In [37]:
# building final cluster summary table
cluster_profiles = (
    cluster_population
    .merge(psych_traits, on="cluster", how="left")
    .merge(media_traits, on="cluster", how="left")
)

cluster_profiles

,cluster,population_share,psych_signals,media_signals
0,0,0.172493,[media_engagement: low (above baseline)],[media_usage: internet_frequency (below baseli...
1,1,0.161806,"[media_engagement: low (below baseline), party...",[media_usage: internet_frequency (above baseli...
2,2,0.053512,"[media_engagement: low (above baseline), media...","[media_usage: tiktok (above baseline), media_u..."
3,3,0.125400,"[media_engagement: low (below baseline), media...",[media_usage: internet_frequency (above baseli...
4,4,0.150883,"[media_engagement: high (below baseline), medi...",[media_usage: internet_frequency (below baseli...
5,5,0.144416,"[media_engagement: low (above baseline), media...",[media_usage: internet_frequency (below baseli...
6,6,0.191490,"[party_alignment: republican (above baseline),...",[media_usage: internet_frequency (below baseli...


In [38]:
output_path = DATA_PROCESSED / "10_mk_cluster_profiles.parquet"

cluster_profiles.to_parquet(output_path, index=False)

print("Saved:", output_path)

Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/10_mk_cluster_profiles.parquet
